# 01_Ingesta_Bronze

Ingesta de los cuatro archivos CSV del proyecto desde el Unity Catalog Volume hacia tablas Delta en la capa Bronze.

El objetivo de esta capa es conservar los datos de negocio sin transformaciones semánticas. Únicamente se agregan metadatos de trazabilidad y se normalizan nombres de columnas para garantizar compatibilidad con Delta Lake.


## 1. Configuración

Se definen el catálogo, el schema Bronze y la ruta del Volume donde se encuentran los archivos raw.


In [ ]:
dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catalog del proyecto")

PROY_CATALOG = dbutils.widgets.get("catalog_name")
PROY_SCHEMA_BRONZE = "bronze"
RAW_PATH = f"/Volumes/{PROY_CATALOG}/{PROY_SCHEMA_BRONZE}/raw_files"

print(f"RAW_PATH: {RAW_PATH}")


## 2. Validación de archivos raw

Antes de ingerir los datos, se verifica que los cuatro archivos fuente estén disponibles en el Volume.


In [ ]:
display(dbutils.fs.ls(RAW_PATH))


## 3. Función reutilizable de ingesta Bronze

La función lee el CSV, mantiene los campos como texto, recupera `_metadata.file_path`, agrega el timestamp de ingesta, normaliza nombres de columnas y escribe una tabla Delta administrada en Unity Catalog.


In [ ]:
from pyspark.sql.functions import current_timestamp, col

def cargar_bronze(nombre_archivo, nombre_tabla, renombres=None):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{RAW_PATH}/{nombre_archivo}")
        .select(
            "*",
            col("_metadata.file_path").alias("_source_file")
        )
        .withColumn("_ingestion_timestamp", current_timestamp())
    )

    if renombres:
        for original, nuevo in renombres.items():
            df = df.withColumnRenamed(original, nuevo)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.{nombre_tabla}"
        )
    )

    print(f"[OK] Tabla creada: {PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.{nombre_tabla}")
    return df


## 4. Normalización técnica de columnas

En Bronze no se modifican los valores de negocio ni se tipan fechas, coordenadas o métricas. Solo se normalizan nombres de columnas para hacerlos compatibles y consistentes en Delta Lake.

La columna `_c0` se conserva intencionalmente en Bronze y se eliminará en Silver.


In [ ]:
renombres_accidentes = {
    "Croquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "hora accidente": "hora_accidente",
    "Gravedad": "gravedad",
    "Clase accidente": "clase_accidente",
    "Localidad": "localidad",
    "Sitio exacto": "sitio_exacto",
    "Nro heridos": "nro_heridos",
    "Numero muertos": "numero_muertos",
    "Placa agente": "placa_agente",
    "Apellidos": "apellidos",
    "Nombres": "nombres",
    "Coordenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y"
}

renombres_causas = {
    "Croquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "hipotesis/causa": "hipotesis_causa"
}

renombres_vehiculos = {
    "Corquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "Hora accidente": "hora_accidente",
    "Direccion": "direccion",
    "Placa": "placa",
    "Clase": "clase",
    "Servicio": "servicio",
    "Clase accidente": "clase_accidente",
    "Corrdenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y",
    "Fecha radica": "fecha_radica"
}

renombres_victimas = {
    "Corquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "Hora accidente": "hora_accidente",
    "Clase": "clase",
    "Direccion": "direccion",
    "Descripcion victima": "descripcion_victima",
    "Gravedad": "gravedad",
    "Id usuario": "id_usuario",
    "Sexo": "sexo",
    "Fecha nacimiento": "fecha_nacimiento",
    "Placa": "placa",
    "Clase.1": "clase_1",
    "Servicio": "servicio",
    "Edad": "edad",
    "Coordenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y"
}


## 5. Ingesta de los cuatro datasets


In [ ]:
df_accidentes_bronze = cargar_bronze(
    "accidentes.csv",
    "accidentes",
    renombres_accidentes
)

df_causas_bronze = cargar_bronze(
    "causas.csv",
    "causas",
    renombres_causas
)

df_vehiculos_bronze = cargar_bronze(
    "vehiculos.csv",
    "vehiculos",
    renombres_vehiculos
)

df_victimas_bronze = cargar_bronze(
    "victimas.csv",
    "victimas",
    renombres_victimas
)


## 6. Validación de conteos

Se comprueba que las tablas Bronze contengan la misma cantidad de registros que los archivos fuente.


In [ ]:
tablas = [
    "accidentes",
    "causas",
    "vehiculos",
    "victimas"
]

for tabla in tablas:
    total = spark.table(
        f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.{tabla}"
    ).count()

    print(f"{tabla}: {total:,}")


Conteos esperados:

- `accidentes`: 23,212
- `causas`: 23,212
- `vehiculos`: 46,128
- `victimas`: 11,328
